# Pseudobulk DEGs: decidual vs non-decidual stromal cells in vivo, compared with in vitro decidualization

Three single-cell datasets, each analysed the same way:

| Dataset | Decidual (`dec`) | Non-decidual (`nonDec`) |
|---|---|---|
| This study | `dec` | `eFib` |
| Ulrich et al. 2024 (20 donors) | decidual cell (dS1-3) | stromal cell of endometrium (eS-proliferative) + fibroblast (C7, RUNX3+) |
| Mareckova et al. 2024 (HECA) | dStromal early / mid / late | eStromal, eStromal cycling, eStromal MMPs, Fibroblast basalis |

Raw counts are summed per donor x group (pseudobulk) and tested with PyDESeq2, paired within donor
(`~donor + condition`). In this study, `timepoint` identifies the donor and `sample` the 10x lanes;
the 3 lanes of each donor are summed, so lanes are never treated as replicates (2 donors, 1 residual
degree of freedom). DEGs use the same cutoffs as the bulk analyses (padj <= 0.01, |FC| >= 2, mean CPM > 3)
and are compared with the in vitro core DEGs (changed in the same direction at 3d, 6d and 8d of cAMP + MPA).

**Outputs** (in `OUT_DIR`):
* `<dataset>_DEGs.csv` - DEGs for each dataset (`<dataset>_all_genes.csv` has every tested gene)
* `InVitroCoreDEGs_vs_singlecell.csv` - in vitro core DEGs with log2FC, p and padj in each single-cell dataset
* `InVitroCoreDEGs_vs_singlecell_stats.csv` - log2FC correlation with each timepoint and Fisher tests of the overlap
* `InVitroCoreDEGs_vs_singlecell_stacked.svg` - % of in vitro core DEGs shared with each dataset
* `InVitroCore_sharedUP_<dataset>.csv` - in vitro core DEGs also up in This study / Ulrich 2024

## 1. Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

plt.rcParams["svg.fonttype"] = "none"      # keep text editable in the SVG
pd.set_option("display.width", 200)

## 2. Datasets

Run `01_DEG_Analysis_Human_InVitro.ipynb` first: section 6 reads its `all_genes_merged.tsv`.

In [ ]:
adata2 = sc.read_h5ad("/directoryto/data/20261111_Human_only_eFib_dec_final.h5ad")
adata3 = sc.read_h5ad("/directoryto/data/Ulrich_OnlyStroma_20Donors.h5ad")
adata4 = sc.read_h5ad("/directoryto/data/endometriumAtlasV2_cells_with_counts.h5ad")

# HECA: one Ensembl ID per gene from the per-study gene_ids columns
idcols = [c for c in adata4.var.columns if c.startswith("gene_ids-")]
adata4.var["ensembl_id"] = adata4.var[idcols].bfill(axis=1).iloc[:, 0].astype(str)

for name, a in [("adata2", adata2), ("adata3", adata3), ("adata4", adata4)]:
    print(f"{name}: {a.n_obs:,} cells x {a.n_vars:,} genes")

## 3. Variables

`counts`: where the raw integer counts are (`("layer", name)` or `("raw", None)`).
`gene_id` / `symbol`: `"index"` for `var_names`, or the name of a `var` column.
Labels not listed in `groups` are excluded.

In [ ]:
# This study: "timepoint" = donor (biological sample); "sample" = 10x lanes, summed into their donor
print(pd.crosstab(adata2.obs["sample"].astype(str), adata2.obs["timepoint"].astype(str)).to_string(), "\n")

DATASETS = {
    "ThisStudy": dict(
        label="This study",
        adata=adata2,
        label_col="cell_type",
        groups={"dec": "dec", "eFib": "nonDec"},
        sample_key="timepoint",                        # donor (2 donors x 3 lanes)
        counts=("layer", "raw_counts"),
        gene_id="gene_ids",
        symbol="index",
    ),
    "Ulrich2024": dict(
        label="Ulrich 2024 (20 donors)",
        adata=adata3,
        label_col="cell_type",
        groups={"decidual cell": "dec",                    # dS1 + dS2 + dS3
                "stromal cell of endometrium": "nonDec",   # eS-proliferative
                "fibroblast": "nonDec"},                   # C7-Fibroblast + RUNX3+_Fibroblast
        sample_key="donor_id",
        counts=("raw", None),
        gene_id="index",
        symbol="feature_name",
    ),
    "Mareckova2024": dict(
        label="Mareckova 2024 (HECA)",
        adata=adata4,
        label_col="celltype",
        groups={"dStromal_early": "dec", "dStromal_mid": "dec", "dStromal_late": "dec",
                "eStromal": "nonDec", "eStromal_cycling": "nonDec",
                "eStromal_MMPs": "nonDec", "Fibroblast_basalis": "nonDec"},
        sample_key="genotype",                         # donor (sample = several per donor)
        counts=("layer", "counts"),
        gene_id="ensembl_id",
        symbol="index",
    ),
}

TEST, REF = "dec", "nonDec"          # log2FC > 0 = higher in decidual cells
MIN_CELLS = 20                       # min cells per donor x group pseudobulk
FDR, FC, MIN_CPM = 0.01, 2.0, 3.0    # padj <= 0.01, |FC| >= 2, mean CPM > 3 (same as bulk and Wang et al. 2020)

BULK_DIR = "/directoryto/BulkRnaSeq"
BULK_MERGED = f"{BULK_DIR}/SharedDEGs_strict/all_genes_merged.tsv"     # from the in vitro notebook
OUT_DIR = Path(f"{BULK_DIR}/SharedDEGs_strict/Human_InVivo_vs_InVitro")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARKERS = ["PRL", "IGFBP1", "FOXO1", "HAND2", "WNT4", "LEFTY2", "PTGER2", "PTGER4", "POSTN"]

## 4. Build pseudobulk counts (sum of raw counts per donor x cell type)

In [ ]:
def build_pseudobulk(cfg):
    a = cfg["adata"]
    group = a.obs[cfg["label_col"]].astype(str).map(cfg["groups"])
    keep = group.notna().values
    obs = pd.DataFrame({"donor": a.obs[cfg["sample_key"]].astype(str).values[keep],
                        "condition": group.values[keep]})

    # raw counts and the matching gene table
    kind, key = cfg["counts"]
    if kind == "raw":
        X, var = a.raw.X, a.raw.var
    else:
        X, var = a.layers[key], a.var
    X = sp.csr_matrix(X)[keep]

    def from_var(spec):
        s = pd.Series(var.index.astype(str), index=var.index) if spec == "index" else \
            (var[spec] if spec in var.columns else a.var[spec].reindex(var.index))
        return s.astype(str).values
    gene_ids = pd.Index(from_var(cfg["gene_id"])).str.split(".").str[0]
    symbols = from_var(cfg["symbol"])

    print(pd.crosstab(obs["donor"], obs["condition"]).T.to_string(), "\n")

    # sum counts per donor x condition
    obs["pb_id"] = obs["donor"] + "__" + obs["condition"]
    codes, pb_ids = pd.factorize(obs["pb_id"])
    onehot = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                           shape=(len(pb_ids), len(codes)))
    pb = pd.DataFrame((onehot @ X).toarray(), index=pb_ids, columns=gene_ids)
    pb = pb.T.groupby(level=0).sum().T                       # merge duplicated gene IDs

    meta = obs.groupby("pb_id").agg(donor=("donor", "first"), condition=("condition", "first"),
                                    n_cells=("donor", "size")).loc[pb.index]
    small = meta.n_cells < MIN_CELLS
    meta, pb = meta[~small], pb[~small]
    print(f"Dropped {small.sum()} pseudobulks with < {MIN_CELLS} cells")

    # paired within donor when possible
    both = meta.groupby("donor")["condition"].nunique()
    paired = (both == 2).sum() >= 2
    if paired:
        meta = meta[meta.donor.isin(both[both == 2].index)]
        pb = pb.loc[meta.index]
    design = "~donor + condition" if paired else "~condition"

    sym_map = pd.Series(symbols, index=gene_ids)
    sym_map = sym_map[~sym_map.index.duplicated()]
    print(f"{len(meta)} pseudobulks from {meta.donor.nunique()} donors; design: {design}")
    print(meta.groupby("condition")["n_cells"].agg(["count", "sum", "median"]).to_string())
    return dict(pb=pb, meta=meta, design=design, sym_map=sym_map)


PB = {}
for name, cfg in DATASETS.items():
    print(f"\n{'=' * 30} {cfg['label']} {'=' * 30}")
    PB[name] = build_pseudobulk(cfg)

## 5. PyDESeq2

In [ ]:
def run_deseq(pbd):
    pb, meta = pbd["pb"], pbd["meta"].copy()
    counts = pb.round().astype(int)
    counts = counts.loc[:, (counts >= 10).sum(axis=0) >= 2]
    meta["condition"] = pd.Categorical(meta["condition"], categories=[REF, TEST])
    meta["donor"] = meta["donor"].astype("category")

    inference = DefaultInference(n_cpus=4)
    dds = DeseqDataSet(counts=counts, metadata=meta, design=pbd["design"],
                       refit_cooks=True, inference=inference, quiet=True)
    dds.deseq2()
    ds = DeseqStats(dds, contrast=["condition", TEST, REF], inference=inference, quiet=True)
    ds.summary()
    res = ds.results_df.copy()
    ds.lfc_shrink(coeff=f"condition[T.{TEST}]")
    res["log2FoldChange_shrunk"] = ds.results_df["log2FoldChange"]

    # expression filter: group-mean CPM
    cpm = pb.div(pb.sum(axis=1), axis=0) * 1e6
    for lvl in (REF, TEST):
        res[f"meanCPM_{lvl}"] = cpm.loc[meta.index[meta["condition"] == lvl], res.index].mean()
    expressed = res[[f"meanCPM_{REF}", f"meanCPM_{TEST}"]].max(axis=1) > MIN_CPM

    sig = (res["padj"] <= FDR) & expressed
    lfc_cut = np.log2(FC)
    res["direction"] = "ns"
    res.loc[sig & (res["log2FoldChange"] >= lfc_cut), "direction"] = "up"
    res.loc[sig & (res["log2FoldChange"] <= -lfc_cut), "direction"] = "down"
    res.insert(0, "gene_symbol", pbd["sym_map"].reindex(res.index).values)
    res.index.name = "gene_id"
    return res.sort_values("padj")


RES = {}
for name, cfg in DATASETS.items():
    print(f"\n{'=' * 30} {cfg['label']} {'=' * 30}")
    res = run_deseq(PB[name])
    RES[name] = res
    res.to_csv(OUT_DIR / f"{name}_all_genes.csv")
    res[res.direction != "ns"].to_csv(OUT_DIR / f"{name}_DEGs.csv")
    print(f"{TEST} vs {REF}: up {(res.direction == 'up').sum()}, down {(res.direction == 'down').sum()} "
          f"(padj <= {FDR}, |FC| >= {FC}, CPM > {MIN_CPM})")
    print(res[res.gene_symbol.isin(MARKERS)][["gene_symbol", "log2FoldChange", "padj", "direction"]].to_string())

## 6. Compare with the in vitro bulk results

In vitro core = DEG in the same direction at 3d, 6d and 8d (cAMP + MPA vs control).

In [ ]:
bulk = pd.read_csv(BULK_MERGED, sep="\t", index_col="gene_id")
TP = ["3d", "6d", "8d"]
dirs = bulk[[f"dir_{t}" for t in TP]]
core_dir = pd.Series("ns", index=bulk.index)
core_dir[(dirs == "up").all(axis=1)] = "up"
core_dir[(dirs == "down").all(axis=1)] = "down"
core = bulk.loc[core_dir != "ns", ["gene_symbol"] + [f"log2FC_{t}" for t in TP] + [f"padj_{t}" for t in TP]].copy()
core.insert(1, "invitro_direction", core_dir[core.index])
print(f"In vitro core DEGs: {len(core)} (up {(core.invitro_direction == 'up').sum()}, "
      f"down {(core.invitro_direction == 'down').sum()})\n")

# one table: core DEGs + each single-cell dataset
table = core.copy()
stats = []
for name, cfg in DATASETS.items():
    res = RES[name]
    r = res.reindex(core.index)
    table[f"{name}_log2FC"] = r["log2FoldChange"]
    table[f"{name}_pvalue"] = r["pvalue"]
    table[f"{name}_padj"] = r["padj"]
    table[f"{name}_direction"] = r["direction"].fillna("not tested")

    # agreement with each timepoint over all genes tested in both
    both = bulk.join(res[["log2FoldChange_shrunk", "direction"]], how="inner")
    row = {"dataset": cfg["label"], "genes_compared": len(both)}
    for t in TP:
        row[f"r_{t}"] = both[[f"log2FC_shrunk_{t}", "log2FoldChange_shrunk"]].corr().iloc[0, 1]
    # overlap of the core with this dataset's DEGs (Fisher, one-sided)
    for d in ("up", "down"):
        a_ = core_dir.reindex(both.index) == d
        b_ = both["direction"] == d
        odds, p = fisher_exact([[(a_ & b_).sum(), (a_ & ~b_).sum()], [(~a_ & b_).sum(), (~a_ & ~b_).sum()]],
                               alternative="greater")
        row[f"core_{d}_shared"] = int((a_ & b_).sum())
        row[f"core_{d}_fisher_p"] = p
    stats.append(row)

table.to_csv(OUT_DIR / "InVitroCoreDEGs_vs_singlecell.csv")
stats = pd.DataFrame(stats).set_index("dataset")
stats.to_csv(OUT_DIR / "InVitroCoreDEGs_vs_singlecell_stats.csv")
print("log2FC correlation with each in vitro timepoint (r), and overlap with the in vitro core:")
print(stats.to_string(formatters={c: (lambda v: f"{v:.2e}") if "fisher_p" in c else
                                  ((lambda v: f"{v:.3f}") if c.startswith("r_") else None)
                                  for c in stats.columns if "fisher_p" in c or c.startswith("r_")}))
print(f"\nSaved {OUT_DIR / 'InVitroCoreDEGs_vs_singlecell.csv'}")

## 7. Plot Results
% of in vitro core DEGs, classified by their result in each single-cell dataset.

In [ ]:
CATS = {
    "Shared, up in both":         "#c0392b",   # red
    "Shared, down in both":       "#f1948a",   # light red
    "Shared, opposite direction": "#4d4d4d",   # dark gray
    "In vitro DEG only":          "#d0d0d0",   # light gray
}

bars = {}
for name, cfg in DATASETS.items():
    b = table["invitro_direction"]
    s = table[f"{name}_direction"].replace("not tested", "ns")
    bars[cfg["label"]] = {
        "Shared, up in both":         ((b == "up") & (s == "up")).sum(),
        "Shared, down in both":       ((b == "down") & (s == "down")).sum(),
        "Shared, opposite direction": ((s != "ns") & (b != s)).sum(),
        "In vitro DEG only":          (s == "ns").sum(),
    }
bar_counts = pd.DataFrame(bars).T[list(CATS)]
bar_pct = bar_counts.div(bar_counts.sum(axis=1), axis=0) * 100
print(bar_counts.assign(total=bar_counts.sum(axis=1)).to_string(), "\n")
print(bar_pct.round(1).to_string())

fig, ax = plt.subplots(figsize=(1.3 * len(bar_pct) + 3, 4.5))
x = np.arange(len(bar_pct))
bottom = np.zeros(len(bar_pct))
for cat, color in CATS.items():
    v = bar_pct[cat].values
    ax.bar(x, v, bottom=bottom, color=color, width=0.65, label=cat)
    for xi, vi, b0 in zip(x, v, bottom):
        if vi >= 3:
            ax.text(xi, b0 + vi / 2, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks(x)
ax.set_xticklabels(bar_pct.index, rotation=20, ha="right")
ax.set_ylabel(f"% of in vitro core DEGs (n = {len(table)})")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "InVitroCoreDEGs_vs_singlecell_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'InVitroCoreDEGs_vs_singlecell_stacked.svg'}")

## 8. In vitro core DEGs also up in vivo

In [ ]:
# In vitro core DEGs that are also UP in each single-cell dataset
show = ["gene_symbol", "log2FC_3d", "log2FC_6d", "log2FC_8d"]

shared_up = {}
for name in ["ThisStudy", "Ulrich2024"]:
    hits = table[(table["invitro_direction"] == "up") & (table[f"{name}_direction"] == "up")]
    hits = hits[show + [f"{name}_log2FC", f"{name}_padj"]].sort_values(f"{name}_log2FC", ascending=False)
    shared_up[name] = hits
    hits.to_csv(OUT_DIR / f"InVitroCore_sharedUP_{name}.csv")
    print(f"\n{'=' * 20} {DATASETS[name]['label']}: {len(hits)} shared up {'=' * 20}")
    print(hits.round(3).to_string())

# Genes up in vitro AND in both single-cell datasets
both = shared_up["ThisStudy"].index.intersection(shared_up["Ulrich2024"].index)
print(f"\nShared up in vitro, This study AND Ulrich 2024: {len(both)}")
print(", ".join(sorted(table.loc[both, "gene_symbol"].astype(str))))